# TCoE Summary Tables

Produces three thesis-ready monetary result tables from the eight solver output CSVs.

| Table | Content |
|---|---|
| **Table 1** | Mean annual TCoE (€/yr) per archetype × strategy, averaged across 7 DSOs |
| **Table 2** | DSO spread per archetype × strategy: max − min TCoE across DSOs |
| **Table 3** | Full TCoE matrix: all archetypes × all DSOs for **tcoe_flex** (the optimal strategy) |

Tables 1 and 2 are combined into one display. Table 3 is exported separately.

**Inputs:** `outputs/results_*.csv`  
**Outputs:** saved next to this notebook in `notebooks/03_analysis/`

**Thesis reference:** Chapter 5, Section 5.1–5.2

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

def find_repo_root(marker='README.md'):
    p = Path('__file__').resolve().parent
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError('Repo root not found')

REPO_ROOT = find_repo_root()
ANALYSIS  = Path('__file__').resolve().parent  # outputs land next to this notebook

# Human-readable archetype labels
ARCHETYPE_LABELS = {
    1: 'A1  Base',
    2: 'A2  Base+PV',
    3: 'A3  Base+PV+BSS',
    4: 'A4  Base+HP',
    5: 'A5  Base+EV',
    6: 'A6  Base+PV+HP',
    7: 'A7  Base+PV+EV',
    8: 'A8  Full Stack',
}
STRATEGY_ORDER = ['no_flex', 'dt_flex', 'tcoe_flex']
DSO_ORDER = ['Westnetz','Bayernwerk','E.DIS','Netze BW','Stromnetz Berlin','SH Netz','MITNETZ STROM']
print(f'Repo root: {REPO_ROOT}')


Repo root: /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub


## Step 1 — Load and consolidate solver outputs

In [2]:
csv_files = sorted((REPO_ROOT / 'outputs').glob('results_*.csv'))
print('Files:', [f.name for f in csv_files])

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    df.columns = df.columns.str.lower()   # normalise case (older CSVs use mixed case)
    dfs.append(df[['household_archetype', 'dso_id', 'strategy', 'total_tcoe_eur']])

all_results = pd.concat(dfs, ignore_index=True)
all_results['archetype_label'] = all_results['household_archetype'].map(ARCHETYPE_LABELS)

print(f'Total rows: {len(all_results)} | expected {8*7*3} = {8*7*3}')
print('Archetypes:', sorted(all_results['household_archetype'].unique()))


Files: ['results_base_2026.csv', 'results_base_ev_2026.csv', 'results_base_hp_2026.csv', 'results_base_pv_2026.csv', 'results_base_pv_bss_2026.csv', 'results_base_pv_bss_hp_ev_2026.csv', 'results_base_pv_ev_2026.csv', 'results_base_pv_hp_2026.csv']
Total rows: 168 | expected 168 = 168
Archetypes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


## Step 2 — Table 1 + 2: Mean TCoE and DSO spread per archetype × strategy

Each cell = mean annual TCoE across all 7 DSOs. The spread (max − min) shows how much the
choice of DSO matters within a given archetype and strategy.

In [3]:
# Compute mean, min, max, spread across DSOs for each (archetype, strategy)
agg = (
    all_results
    .groupby(['household_archetype', 'archetype_label', 'strategy'])['total_tcoe_eur']
    .agg(mean='mean', min='min', max='max')
    .reset_index()
)
agg['spread'] = agg['max'] - agg['min']

# Identify cheapest and most expensive DSO per (archetype, strategy)
def cheapest_dso(grp): return grp.loc[grp['total_tcoe_eur'].idxmin(), 'dso_id']
def costliest_dso(grp): return grp.loc[grp['total_tcoe_eur'].idxmax(), 'dso_id']
dso_extremes = (
    all_results
    .groupby(['household_archetype', 'strategy'])
    .apply(lambda g: pd.Series({'cheapest_dso': cheapest_dso(g), 'costliest_dso': costliest_dso(g)}))
    .reset_index()
)
agg = agg.merge(dso_extremes, on=['household_archetype', 'strategy'])

# Pivot mean into archetype × strategy matrix
t1_mean = agg.pivot_table(
    index='archetype_label', columns='strategy', values='mean', aggfunc='first'
)[STRATEGY_ORDER].round(0).astype(int)
t1_mean.columns = ['no_flex (€/yr)', 'dt_flex (€/yr)', 'tcoe_flex (€/yr)']
t1_mean.index.name = 'Archetype'

# Pivot spread (max-min) into archetype × strategy matrix
t2_spread = agg.pivot_table(
    index='archetype_label', columns='strategy', values='spread', aggfunc='first'
)[STRATEGY_ORDER].round(0).astype(int)
t2_spread.columns = ['spread no_flex (€)', 'spread dt_flex (€)', 'spread tcoe_flex (€)']
t2_spread.index.name = 'Archetype'

# Combined view
combined = t1_mean.join(t2_spread)
print('Table 1+2 — Mean TCoE (€/yr) and DSO spread (max−min €) per archetype × strategy:')
display(combined)


Table 1+2 — Mean TCoE (€/yr) and DSO spread (max−min €) per archetype × strategy:


/var/folders/2h/6b37_vh54r522sl857g198hw0000gn/T/ipykernel_80758/2865201421.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({'cheapest_dso': cheapest_dso(g), 'costliest_dso': costliest_dso(g)}))


,no_flex (€/yr),dt_flex (€/yr),tcoe_flex (€/yr),spread no_flex (€),spread dt_flex (€),spread tcoe_flex (€)
Archetype,,,,,,
A1 Base,1310,1310,1310,236,236,236
A2 Base+PV,307,307,307,133,133,133
A3 Base+PV+BSS,7,-76,-170,77,95,77
A4 Base+HP,2459,2313,2118,475,481,217
A5 Base+EV,2317,1881,1763,405,406,231
A6 Base+PV+HP,1333,1171,765,368,368,150
A7 Base+PV+EV,1195,756,431,299,300,103
A8 Full Stack,2006,1367,683,437,485,193


## Step 3 — Table 3: Full TCoE matrix for tcoe_flex

Rows = archetypes, columns = DSOs. Shows the absolute annual electricity cost
after full-stack optimisation for each archetype–DSO combination.

In [4]:
tcoe_flex_only = all_results[all_results['strategy'] == 'tcoe_flex'].copy()

t3 = tcoe_flex_only.pivot_table(
    index='archetype_label', columns='dso_id', values='total_tcoe_eur', aggfunc='first'
)
# Reorder columns to consistent DSO order (keep any extras at end)
ordered_cols = [d for d in DSO_ORDER if d in t3.columns]
t3 = t3[ordered_cols].round(0).astype(int)
t3.index.name = 'Archetype'
t3.columns.name = None

# Add mean and spread columns for quick reference
t3['Mean'] = t3.mean(axis=1).round(0).astype(int)
t3['Spread'] = (t3[ordered_cols].max(axis=1) - t3[ordered_cols].min(axis=1)).astype(int)

print('Table 3 — TCoE tcoe_flex (€/yr) by archetype × DSO:')
display(t3)


Table 3 — TCoE tcoe_flex (€/yr) by archetype × DSO:


,Westnetz,Bayernwerk,E.DIS,Netze BW,Stromnetz Berlin,SH Netz,MITNETZ STROM,Mean,Spread
Archetype,,,,,,,,,
A1 Base,1461,1225,1239,1360,1294,1311,1280,1310,236
A2 Base+PV,398,267,265,340,277,316,287,307,133
A3 Base+PV+BSS,-137,-166,-184,-153,-215,-153,-180,-170,78
A4 Base+HP,2228,2077,2011,2224,2136,2070,2079,2118,217
A5 Base+EV,1896,1665,1697,1808,1776,1740,1757,1763,231
A6 Base+PV+HP,803,788,699,849,744,738,732,765,150
A7 Base+PV+EV,496,418,393,449,412,435,416,431,103
A8 Full Stack,651,775,612,805,662,636,642,683,193


## Step 4 — Bonus: strategy comparison per archetype (mean saving vs. no_flex)

In [5]:
# How much does each strategy save vs. no_flex on average across DSOs?
means = agg.pivot_table(
    index='archetype_label', columns='strategy', values='mean', aggfunc='first'
)[STRATEGY_ORDER]

savings = pd.DataFrame(index=means.index)
savings['no_flex (€/yr)']          = means['no_flex'].round(0).astype(int)
savings['dt_flex (€/yr)']          = means['dt_flex'].round(0).astype(int)
savings['tcoe_flex (€/yr)']        = means['tcoe_flex'].round(0).astype(int)
savings['Δ dt_flex vs no_flex (€)']   = (means['no_flex'] - means['dt_flex']).round(0).astype(int)
savings['Δ tcoe_flex vs no_flex (€)'] = (means['no_flex'] - means['tcoe_flex']).round(0).astype(int)
savings['Δ tcoe vs dt_flex (€)']       = (means['dt_flex'] - means['tcoe_flex']).round(0).astype(int)
savings.index.name = 'Archetype'

print('Table 4 — Mean annual savings vs. no_flex baseline:')
display(savings)


Table 4 — Mean annual savings vs. no_flex baseline:


,no_flex (€/yr),dt_flex (€/yr),tcoe_flex (€/yr),Δ dt_flex vs no_flex (€),Δ tcoe_flex vs no_flex (€),Δ tcoe vs dt_flex (€)
Archetype,,,,,,
A1 Base,1310,1310,1310,0,0,0
A2 Base+PV,307,307,307,0,0,0
A3 Base+PV+BSS,7,-76,-170,83,177,94
A4 Base+HP,2459,2313,2118,146,341,195
A5 Base+EV,2317,1881,1763,435,554,118
A6 Base+PV+HP,1333,1171,765,162,568,406
A7 Base+PV+EV,1195,756,431,438,764,325
A8 Full Stack,2006,1367,683,639,1322,684


## Step 5 — Export

In [6]:
# Table 1+2: mean + spread
out1 = ANALYSIS / 'tcoe_mean_spread_by_archetype_strategy_2026.csv'
combined.to_csv(out1)
print(f'Saved: {out1.name}')

# Table 3: full tcoe_flex matrix
out2 = ANALYSIS / 'tcoe_flex_matrix_archetype_dso_2026.csv'
t3.to_csv(out2)
print(f'Saved: {out2.name}')

# Table 4: strategy comparison
out3 = ANALYSIS / 'tcoe_strategy_savings_2026.csv'
savings.to_csv(out3)
print(f'Saved: {out3.name}')


Saved: tcoe_mean_spread_by_archetype_strategy_2026.csv
Saved: tcoe_flex_matrix_archetype_dso_2026.csv
Saved: tcoe_strategy_savings_2026.csv
